# LLM Fundamentals & APIs Assignment

This notebook demonstrates practical implementation of multi-LLM comparison, prompt engineering, streaming chat, and token tracking using the **Groq** and **Google Gemini** APIs.

In [15]:
import os
import time
import json
import pandas as pd
import urllib.request
from dotenv import load_dotenv
import tiktoken
from groq import Groq
import google.generativeai as genai

# Load API keys from .env file
load_dotenv()

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

# Initialize Clients
groq_client = Groq(api_key=GROQ_API_KEY)
genai.configure(api_key=GEMINI_API_KEY)

print("Clients initialized.")

Clients initialized.


## Optional Dataset Preparation
We will download a sample of the BBC News dataset for our experiments.

In [16]:
import urllib.request
import json

# We will use a direct raw URL to a JSON formatted subset of BBC News to avoid Kaggle CLI requirements
# If it fails, we will use a fallback sample article.
bbc_url = "https://raw.githubusercontent.com/suraj-deshmukh/BBC-Dataset-News-Classification/master/dataset/dataset.csv"
try:
    # Reading first 100 rows just for efficiency
    df_bbc = pd.read_csv(bbc_url, nrows=100)
    sample_article = df_bbc['news'].iloc[0]
    print("Successfully loaded BBC News dataset.")
except Exception as e:
    print(f"Could not load from URL, using fallback article. Error: {e}")
    sample_article = """Apple Inc. reported its fiscal third-quarter results on Thursday, beating Wall Street expectations for both revenue and earnings. The tech giant's performance was driven by strong sales of its services and a better-than-expected showing for the iPhone. Apple's total revenue for the quarter was $81.8 billion, down 1% from the same period last year, but slightly above analysts' estimates. Earnings per share came in at $1.26, exceeding the $1.19 projected by analysts. The company's Services business, which includes the App Store, Apple Music, and Apple Pay, generated a record $21.2 billion in revenue, up 8% year-over-year."""

# Let's see the start of our text
print("\nSample Article Snippet:")
print(sample_article[:300] + "...")

Successfully loaded BBC News dataset.

Sample Article Snippet:
China had role in Yukos split-up
 
 China lent Russia $6bn (£3.2bn) to help the Russian government renationalise the key Yuganskneftegas unit of oil group Yukos, it has been revealed.
 
 The Kremlin said on Tuesday that the $6bn which Russian state bank VEB lent state-owned Rosneft to help buy Yugan...


## Task 1: Multi-LLM Response Comparison Tool
We will send the same prompt to Groq (Llama-3) and Google Gemini (Gemini-1.5-Flash), storing the responses and calculating the time and length.

In [17]:
def call_groq(prompt, model="llama-3.1-8b-instant"):
    start = time.time()
    try:
        response = groq_client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model=model,
        )
        content = response.choices[0].message.content
    except Exception as e:
        content = f"Groq Error: {str(e)}"
    end = time.time()
    return content, end - start, len(content)

def call_gemini(prompt):
    start = time.time()
    try:
        model = genai.GenerativeModel('gemini-1.5-flash')
        response = model.generate_content(prompt)
        content = response.text
    except Exception as e:
        content = f"Gemini Error: {str(e)}"
        # Fallback to another Groq model if Gemini key is invalid to fulfill the 2-model requirement
        print("Gemini failed, falling back to Groq Mixtral for comparison.")
        return call_groq(prompt, model="mixtral-8x7b-32768")
        
    end = time.time()
    return content, end - start, len(content)

In [18]:
prompt = f"Please write a 2-sentence summary of the following article:\n\n{sample_article[:1000]}"

print("Calling Groq Llama-3...")
groq_resp, groq_time, groq_len = call_groq(prompt)

print("Calling Gemini (or Fallback)...")
gemini_resp, gemini_time, gemini_len = call_gemini(prompt)

comparison_data = [
    {"Provider": "Groq (Llama-3)", "Response Time (s)": round(groq_time, 2), "Length (chars)": groq_len, "Output": groq_resp},
    {"Provider": "Gemini / Fallback", "Response Time (s)": round(gemini_time, 2), "Length (chars)": gemini_len, "Output": gemini_resp}
]

# Display in a DataFrame
df_compare = pd.DataFrame(comparison_data)
df_compare.to_csv('llm_comparisons.csv', index=False)
print("Saved comparison to llm_comparisons.csv")
display(df_compare)

Calling Groq Llama-3...
Calling Gemini (or Fallback)...
Gemini failed, falling back to Groq Mixtral for comparison.
Saved comparison to llm_comparisons.csv


,Provider,Response Time (s),Length (chars),Output
0,Groq (Llama-3),0.70,446,Here's a 2-sentence summary of the article:\n\...
1,Gemini / Fallback,0.11,314,Groq Error: Error code: 400 - {'error': {'mess...


## Task 2: Prompt Engineering Playground
Testing 5 different prompts for summarizing the same use case using Groq.

In [19]:
prompts = {
    "Prompt 1 (Basic)": "Summarize this article: ",
    "Prompt 2 (Bullets)": "Extract the 3 most important key points from this article in a bulleted list: ",
    "Prompt 3 (ELI5)": "Explain this article as if I am 5 years old: ",
    "Prompt 4 (Entities)": "List all the people, companies, and locations mentioned in this article: ",
    "Prompt 5 (Tweet)": "Summarize this article into a single engaging tweet with hashtags: "
}

results = {}

for name, p_text in prompts.items():
    print(f"Running {name}...")
    full_prompt = p_text + "\n" + sample_article[:1000]
    content, _, _ = call_groq(full_prompt)
    results[name] = {"prompt": p_text, "output": content}
    
# Save to JSON
with open('prompt_engineering_results.json', 'w') as f:
    json.dump(results, f, indent=4)
print("\nSaved results to prompt_engineering_results.json")

for name, res in results.items():
    print(f"\n--- {name} ---")
    print(res['output'])

Running Prompt 1 (Basic)...
Running Prompt 2 (Bullets)...
Running Prompt 3 (ELI5)...
Running Prompt 4 (Entities)...
Running Prompt 5 (Tweet)...

Saved results to prompt_engineering_results.json

--- Prompt 1 (Basic) ---
The article reveals that China had a significant role in the split-up of the Russian oil company Yukos. 

Here are the key points:

1. Russia's state-owned bank VEB lent $6 billion to Rosneft to help buy the key unit Yuganskneftegas from Yukos. 
2. VEB stated that the $6 billion loan came from Chinese banks.
3. In return, Rosneft signed a long-term oil supply deal with China's state-owned oil company CNPC, receiving $6 billion in credits from them.
4. The credits from CNPC were used to pay off the loans Rosneft received to finance the purchase of Yugansk.
5. CNPC initially offered 20% of Yugansk in exchange for financing but opted for a long-term oil supply deal instead, likely to avoid possible litigation from Yukos.
6. This deal suggests a significant Chinese involvem

**Prompt Engineering Analysis:**
- `Prompt 2 (Bullets)` provides the most structured and scannable summary.
- `Prompt 3 (ELI5)` is excellent for making complex news accessible.
- `Prompt 5 (Tweet)` is the best for social media content generation.
Overall, `Prompt 2` gave the most universally useful result for quick information retrieval.

## Task 3: Streaming AI Chat Assistant
Implementing a chatbot using the Groq API with `stream=True` to print tokens chunk-by-chunk.

In [20]:
def streaming_chat(user_input, chat_history=None):
    if chat_history is None:
        chat_history = [{"role": "system", "content": "You are a helpful and concise AI assistant."}]
        
    chat_history.append({"role": "user", "content": user_input})
    
    print(f"User: {user_input}\nAssistant: ", end="")
    try:
        stream = groq_client.chat.completions.create(
            messages=chat_history,
            model="llama-3.1-8b-instant",
            stream=True
        )
        
        assistant_response = ""
        for chunk in stream:
            if chunk.choices[0].delta.content is not None:
                token = chunk.choices[0].delta.content
                print(token, end="", flush=True)
                assistant_response += token
        print("\n")
        chat_history.append({"role": "assistant", "content": assistant_response})
    except Exception as e:
        print(f"\n[Streaming Error: {e}]")
        
    return chat_history

# Simulated Chat
history = streaming_chat("What is the capital of France?")
history = streaming_chat("And what is a famous landmark there?", history)

User: What is the capital of France?
Assistant: The capital of France is Paris.

User: And what is a famous landmark there?
Assistant: A famous landmark in Paris is the Eiffel Tower (Tour Eiffel in French). It's an iconic iron lattice tower built in 1889 for the World's Fair and is a symbol of the city.



## Task 4: Token Usage and Cost Tracker
Creating a utility to track prompt text, response text, tokens, and estimated cost. We will use `tiktoken` to estimate tokens.

In [21]:
class TokenTracker:
    def __init__(self):
        self.logs = []
        # Using cl100k_base which is standard for GPT-3.5/4 (close enough estimation for others)
        self.encoding = tiktoken.get_encoding("cl100k_base")
        # Assume pricing per 1M tokens (e.g., standard API costs)
        self.input_cost_per_m = 0.50 
        self.output_cost_per_m = 1.50
        
    def count_tokens(self, text):
        return len(self.encoding.encode(text))
        
    def log_call(self, prompt, response):
        prompt_tokens = self.count_tokens(prompt)
        response_tokens = self.count_tokens(response)
        total_tokens = prompt_tokens + response_tokens
        
        cost = (prompt_tokens / 1_000_000 * self.input_cost_per_m) + \
               (response_tokens / 1_000_000 * self.output_cost_per_m)
               
        self.logs.append({
            "Prompt": prompt[:50] + "...",
            "Response Length": len(response),
            "Prompt Tokens": prompt_tokens,
            "Response Tokens": response_tokens,
            "Total Tokens": total_tokens,
            "Estimated Cost ($)": round(cost, 6)
        })
        
    def generate_report(self):
        df = pd.DataFrame(self.logs)
        df.to_csv('token_usage_logs.csv', index=False)
        
        total_requests = len(df)
        total_tokens = df['Total Tokens'].sum()
        total_cost = df['Estimated Cost ($)'].sum()
        
        print(f"\n--- FINAL USAGE REPORT ---")
        print(f"Total Requests: {total_requests}")
        print(f"Total Tokens Used: {total_tokens}")
        print(f"Total Estimated Cost: ${total_cost:.6f}")
        print("Logs saved to token_usage_logs.csv")
        return df

# Test the tracker
tracker = TokenTracker()
prompt = "Write a short poem about AI."
resp, _, _ = call_groq(prompt)
tracker.log_call(prompt, resp)

prompt2 = "What is Python?"
resp2, _, _ = call_groq(prompt2)
tracker.log_call(prompt2, resp2)

display(tracker.generate_report())


--- FINAL USAGE REPORT ---
Total Requests: 2
Total Tokens Used: 603
Total Estimated Cost: $0.000893
Logs saved to token_usage_logs.csv


,Prompt,Response Length,Prompt Tokens,Response Tokens,Total Tokens,Estimated Cost ($)
0,Write a short poem about AI....,596,7,149,156,0.000227
1,What is Python?...,2271,4,443,447,0.000666
